In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import joblib
import datetime
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Load Processed Data

In [2]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

print("VENLIX AI - Last Mile Delivery Failure Predictor")
print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

VENLIX AI - Last Mile Delivery Failure Predictor
Training Shape : (34918, 27)
Testing Shape  : (8730, 27)


# 2. Verify Features

In [3]:
if list(X_train.columns) != list(X_test.columns):
    raise ValueError("Feature mismatch between X_train and X_test!")
print("Train and test features match perfectly.")

Train and test features match perfectly.


# 3. Model Training & Comparison

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=20, min_samples_leaf=10, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_split=10, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, scale_pos_weight=(y_train == 0).sum()/(y_train == 1).sum(), eval_metric="logloss"),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, class_weight="balanced")
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else [0]*len(X_test)
    
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring="f1").mean()
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred),
        "ROC AUC": roc_auc_score(y_test, prob) if hasattr(model, "predict_proba") else 0.0,
        "CV F1": cv_score,
        "model_obj": model
    })

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != "model_obj"} for r in results])
results_df = results_df.sort_values("F1 Score", ascending=False).reset_index(drop=True)
display(results_df)

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Number of positive: 6823, number of negative: 28095
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 34918, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number of positive: 5458, number of negative: 22476
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003320 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,CV F1
0,LightGBM,0.999542,0.998244,0.999414,0.998828,0.999997,0.999707
1,XGBoost,0.999542,0.998244,0.999414,0.998828,0.999999,0.999560
2,Decision Tree,0.995762,0.982091,0.996483,0.989235,0.999024,0.988795
3,Random Forest,0.995533,0.979298,0.998242,0.988679,0.999933,0.981866
4,Logistic Regression,0.890722,0.660410,0.907386,0.764444,0.958471,0.752388


Training Random Forest...


Training XGBoost...


Training LightGBM...


...

[LightGBM] [Info] Number of positive: 5458, number of negative: 22476
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003801 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 841
[LightGBM] [Info] Number of data points in the train set: 27934, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[LightGBM] [Info] Number of positive: 5458, number of negative: 22476
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 27934, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[LightGBM] [Info] Number of positive: 5459, number of negative: 22476
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003031 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 27935, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[LightGBM] [Info] Number of positive: 5459, number of negative: 22476
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002637 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 27935, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC,CV F1
0,LightGBM,0.999542,0.998244,0.999414,0.998828,0.999997,0.999707
1,XGBoost,0.999542,0.998244,0.999414,0.998828,0.999999,0.999560
2,Decision Tree,0.995762,0.982091,0.996483,0.989235,0.999024,0.988795
3,Random Forest,0.995533,0.979298,0.998242,0.988679,0.999933,0.981866
4,Logistic Regression,0.890722,0.660410,0.907386,0.764444,0.958471,0.752388


# 4. Select and Validate Best Model

In [6]:
best_row = results_df.iloc[0]
best_model_name = best_row["Model"]
best_model = next(r["model_obj"] for r in results if r["Model"] == best_model_name)

print(f"Best Model automatically selected: {best_model_name} (F1 Score: {best_row['F1 Score']:.4f})")

# Validating feature names matching
# The exact check required: model.get_booster().feature_names equals list(X_test.columns)
# Fallback implemented for non-XGBoost models just in case.
if hasattr(best_model, "get_booster"):
    model_features = best_model.get_booster().feature_names
elif hasattr(best_model, "feature_names_in_"):
    model_features = list(best_model.feature_names_in_)
else:
    model_features = list(X_test.columns)

if list(model_features) != list(X_test.columns):
    print("Features in model:", model_features)
    print("Features in X_test:", list(X_test.columns))
    raise ValueError("Model features do not match X_test.columns. Stop training.")
else:
    print("Feature validation passed.")

Best Model automatically selected: LightGBM (F1 Score: 0.9988)
Feature validation passed.


# 5. Save Model and Metadata

In [7]:
# 1. Save Model
joblib.dump(best_model, "../models/venlix_model.pkl")

# 2. Save Feature Columns
feature_columns = list(X_train.columns)
joblib.dump(feature_columns, "../models/feature_columns.pkl")

# 3. Save model_metadata.json
metadata = {
    "feature_names": feature_columns,
    "best_model": best_model_name,
    "accuracy": best_row["Accuracy"],
    "precision": best_row["Precision"],
    "recall": best_row["Recall"],
    "f1": best_row["F1 Score"],
    "roc_auc": best_row["ROC AUC"],
    "training_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "number_of_features": len(feature_columns)
}

with open("../models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("venlix_model.pkl, feature_columns.pkl, and model_metadata.json saved successfully in ../models/")

venlix_model.pkl, feature_columns.pkl, and model_metadata.json saved successfully in ../models/


In [8]:
import joblib

model = joblib.load("venlix_model.pkl")

print("Number of features:", len(model.get_booster().feature_names))

print("\nModel Features:")

for f in model.get_booster().feature_names:
    print(f)

Number of features: 27

Model Features:
Agent_Age
Agent_Rating
Weather
Traffic
Vehicle
Area
Delivery_Time
customer_answered_call
customer_response_time
customer_availability
visitor_pass_status
society_security_level
gate_wait_time
driver_status
previous_failed_deliveries
address_confidence
preferred_delivery_slot
estimated_arrival_delay
driver_experience
pickup_delay_minutes
hour_of_day
day_of_week
is_weekend
arrival_within_preferred_slot
customer_reachability_score
society_accessibility_score
driver_reliability_score
